# Amazon Search / LTR — Week 4 V1 One-Click Colab
## SBERT Embeddings → FAISS Vector Retrieval → Top-50 → BM25 Comparison

This notebook is the **Week 4 V1 text-only vector retrieval baseline**.

### What it does automatically
1. Installs dependencies.
2. Uploads and extracts the Week 1 + Week 2 ZIP files.
3. Loads the 10k cleaned Amazon catalog.
4. Reuses the fixed Week 2 test-query IDs.
5. Builds a fresh full-catalog BM25 reference.
6. Encodes `product_text` with Sentence-BERT.
7. Builds a FAISS cosine-similarity index.
8. Retrieves Top-50 products for each test query.
9. Evaluates:
   - **Judged-candidate NDCG@10**
   - **Judged-candidate MRR**
   - **Full-catalog known-relevant Recall@10**
   - **Full-catalog known-relevant Recall@50**
10. Measures BM25, SBERT query-embedding, and FAISS search latency.
11. Exports embeddings, FAISS index, Top-50 results, metrics, plots, and a ZIP bundle.

### Expected input files
Upload these two files when prompted:
- `amazon_week1_v4_processed_data.zip` (filename may include `(1)`)
- `amazon_week2_baseline_ranking_results.zip` (filename may include `(1)`)

The notebook searches by the files **inside** the ZIPs, so minor ZIP filename changes are okay.

> **Important:** This V1 intentionally finishes the reliable text-vector baseline first.  
> CLIP image embeddings + multimodal fusion should be added as Week 4 V2 after this pipeline is stable.

In [ ]:
# ============================================================
# 0. ONE-CLICK CONFIG
# ============================================================

TOP_K = 50
KNOWN_RELEVANT_THRESHOLD = 2
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SBERT_BATCH_SIZE = 128
MAX_TEST_QUERIES = None
LATENCY_SAMPLE_SIZE = 30
RANDOM_SEED = 42

print("Week 4 V1 configuration loaded.")
print({
    "TOP_K": TOP_K,
    "KNOWN_RELEVANT_THRESHOLD": KNOWN_RELEVANT_THRESHOLD,
    "SBERT_MODEL": SBERT_MODEL,
    "MAX_TEST_QUERIES": MAX_TEST_QUERIES,
    "LATENCY_SAMPLE_SIZE": LATENCY_SAMPLE_SIZE
})

In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

import sys, subprocess

packages = [
    "sentence-transformers>=3.0",
    "faiss-cpu>=1.8",
    "rank-bm25>=0.2.2",
    "pyarrow>=15",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies installed.")

In [ ]:
# ============================================================
# 2. IMPORTS + ENVIRONMENT
# ============================================================

import os, re, gc, json, math, time, shutil, random, zipfile, sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import ndcg_score
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import torch

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
INPUT_DIR = ROOT / "week4_inputs"
EXTRACT_DIR = ROOT / "week4_extracted"
OUTPUT_DIR = ROOT / "week4_v1_outputs"

for p in [INPUT_DIR, EXTRACT_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Working directory:", ROOT)

In [ ]:
# ============================================================
# 3. UPLOAD WEEK 1 + WEEK 2 ZIP FILES
# ============================================================

is_colab = False
try:
    from google.colab import files
    is_colab = True
except Exception:
    files = None

if is_colab:
    print("Please upload BOTH the Week 1 and Week 2 ZIP files.")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        target = INPUT_DIR / filename
        with open(target, "wb") as f:
            f.write(content)
        print("Saved:", target)
else:
    print("Not running in Colab.")
    print("Place the Week 1 and Week 2 ZIP files in:", INPUT_DIR)

zip_files = sorted(INPUT_DIR.glob("*.zip"))

if len(zip_files) < 2:
    for z in ROOT.glob("*.zip"):
        if z.parent != INPUT_DIR and "week4_v1_outputs" not in z.name.lower():
            copied = INPUT_DIR / z.name
            if not copied.exists():
                shutil.copy2(z, copied)
    zip_files = sorted(INPUT_DIR.glob("*.zip"))

print("\nZIP files found:")
for z in zip_files:
    print(" -", z.name)

if len(zip_files) < 2:
    raise FileNotFoundError("Need the Week 1 and Week 2 ZIP files. Upload both and rerun this cell.")

In [ ]:
# ============================================================
# 4. EXTRACT + AUTO-DETECT REQUIRED FILES
# ============================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

for zpath in zip_files:
    out = EXTRACT_DIR / zpath.stem
    out.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(out)
        print("Extracted:", zpath.name)
    except zipfile.BadZipFile:
        print("Skipping corrupted/non-ZIP:", zpath)

def find_one(filename):
    matches = list(EXTRACT_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find required file: {filename}")
    return matches[0]

PRODUCTS_PATH = find_one("cleaned_products_10k.parquet")
QPR_PATH = find_one("query_product_relevance.parquet")
FIXED_TEST_PATH = find_one("fixed_test_queries.csv")

benchmark_paths = list(EXTRACT_DIR.rglob("benchmark_metrics.csv"))
WEEK2_BENCHMARK_PATH = benchmark_paths[0] if benchmark_paths else None

print("\nDetected:")
print("Products:", PRODUCTS_PATH)
print("Query-product relevance:", QPR_PATH)
print("Fixed test queries:", FIXED_TEST_PATH)
print("Week 2 benchmark:", WEEK2_BENCHMARK_PATH)

In [ ]:
# ============================================================
# 5. LOAD + VALIDATE DATA
# ============================================================

products = pd.read_parquet(PRODUCTS_PATH)
qpr = pd.read_parquet(QPR_PATH)
fixed_test = pd.read_csv(FIXED_TEST_PATH)

products["product_id"] = products["product_id"].astype(str)
qpr["product_id"] = qpr["product_id"].astype(str)
qpr["query_id"] = qpr["query_id"].astype(str)
fixed_test["query_id"] = fixed_test["query_id"].astype(str)

products = products.drop_duplicates("product_id").reset_index(drop=True)

required_product_cols = {"product_id", "title"}
required_qpr_cols = {"query_id", "query", "product_id", "relevance_label"}

missing_p = required_product_cols - set(products.columns)
missing_q = required_qpr_cols - set(qpr.columns)

if missing_p:
    raise ValueError(f"Products file missing columns: {missing_p}")
if missing_q:
    raise ValueError(f"Relevance file missing columns: {missing_q}")

def safe_col(df, col):
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)

existing_product_text = safe_col(products, "product_text").str.strip()

reconstructed_text = (
    safe_col(products, "title").str.strip() + " " +
    safe_col(products, "brand").str.strip() + " " +
    safe_col(products, "description").str.strip() + " " +
    safe_col(products, "features").str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

products["retrieval_text"] = np.where(
    existing_product_text.str.len() > 0,
    existing_product_text,
    reconstructed_text
)
products["title"] = safe_col(products, "title")

query_map = (
    qpr[["query_id", "query"]]
    .drop_duplicates("query_id")
    .set_index("query_id")["query"]
    .to_dict()
)

test_ids = [qid for qid in fixed_test["query_id"].tolist() if qid in query_map]
if MAX_TEST_QUERIES is not None:
    test_ids = test_ids[:MAX_TEST_QUERIES]

test_queries = pd.DataFrame({
    "query_id": test_ids,
    "query": [query_map[qid] for qid in test_ids]
})

gold = qpr[qpr["query_id"].isin(test_ids)].copy()

print("Catalog products:", len(products))
print("Week 1 relevance rows:", len(qpr))
print("Fixed test queries used:", len(test_queries))
print("Judged rows for test queries:", len(gold))
print("Average judged candidates/query:", round(len(gold) / max(len(test_queries), 1), 2))

display(test_queries.head())
display(products[["product_id", "title", "retrieval_text"]].head(3))

In [ ]:
# ============================================================
# 6. EVALUATION HELPERS
# ============================================================

product_ids = products["product_id"].tolist()
pid_to_idx = {pid: i for i, pid in enumerate(product_ids)}
idx_to_pid = np.array(product_ids, dtype=object)
title_map = products.set_index("product_id")["title"].to_dict()

relevance_lookup = defaultdict(dict)
for row in gold[["query_id", "product_id", "relevance_label"]].itertuples(index=False):
    relevance_lookup[row.query_id][row.product_id] = int(row.relevance_label)

known_positives = (
    gold[gold["relevance_label"] >= KNOWN_RELEVANT_THRESHOLD]
    .groupby("query_id")["product_id"]
    .apply(lambda x: set(x.astype(str)))
    .to_dict()
)

def recall_at_k(retrieved_ids, positive_ids, k):
    positive_ids = set(positive_ids)
    if not positive_ids:
        return np.nan
    return len(set(retrieved_ids[:k]) & positive_ids) / len(positive_ids)

def mrr_from_labels(labels_in_rank_order):
    for rank, label in enumerate(labels_in_rank_order, start=1):
        if label > 0:
            return 1.0 / rank
    return 0.0

def judged_metrics_for_scores(query_id, score_by_pid, k=10):
    rows = gold[gold["query_id"] == query_id][["product_id", "relevance_label"]].copy()
    rows["score"] = rows["product_id"].map(score_by_pid)
    rows["score"] = rows["score"].fillna(-1e9)

    labels = rows["relevance_label"].astype(float).to_numpy()
    scores = rows["score"].astype(float).to_numpy()

    if len(rows) == 0:
        return np.nan, np.nan

    ndcg = float(ndcg_score([labels], [scores], k=min(k, len(labels))))
    ranked = rows.sort_values("score", ascending=False)
    mrr = mrr_from_labels(ranked["relevance_label"].tolist())
    return ndcg, mrr

def summarize_query_metrics(df):
    return {
        "NDCG@10_judged": df["NDCG@10_judged"].mean(),
        "MRR_judged": df["MRR_judged"].mean(),
        "Recall@10_full": df["Recall@10_full"].mean(),
        "Recall@50_full": df["Recall@50_full"].mean(),
        "queries_evaluated": len(df),
    }

print("Evaluation helpers ready.")

In [ ]:
# ============================================================
# 7. WEEK 2 REFERENCE METRICS
# ============================================================

if WEEK2_BENCHMARK_PATH is not None:
    week2_reference = pd.read_csv(WEEK2_BENCHMARK_PATH)
    print("Original Week 2 benchmark file:")
    display(week2_reference)
else:
    week2_reference = None
    print("Week 2 benchmark_metrics.csv not found; continuing.")

In [ ]:
# ============================================================
# 8. BM25 FULL-CATALOG REFERENCE
# ============================================================

TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:[._/-][a-z0-9]+)*", re.I)

def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())

print("Tokenizing 10k catalog...")
tokenized_corpus = [tokenize(x) for x in products["retrieval_text"].tolist()]

print("Building BM25 index...")
bm25 = BM25Okapi(tokenized_corpus)

bm25_top_rows = []
bm25_query_metrics = []
bm25_latency_ms = []

for n, row in enumerate(test_queries.itertuples(index=False), start=1):
    qid, query = row.query_id, row.query

    t0 = time.perf_counter()
    scores = np.asarray(bm25.get_scores(tokenize(query)), dtype=np.float32)
    top_idx = np.argpartition(-scores, min(TOP_K, len(scores)) - 1)[:TOP_K]
    top_idx = top_idx[np.argsort(-scores[top_idx])]
    latency = (time.perf_counter() - t0) * 1000.0
    bm25_latency_ms.append(latency)

    retrieved_pids = idx_to_pid[top_idx].tolist()

    for rank, idx in enumerate(top_idx, start=1):
        pid = str(idx_to_pid[idx])
        bm25_top_rows.append({
            "model": "BM25",
            "query_id": qid,
            "query": query,
            "rank": rank,
            "product_id": pid,
            "title": title_map.get(pid, ""),
            "relevance_label": relevance_lookup[qid].get(pid, np.nan),
            "score": float(scores[idx]),
        })

    judged_pids = gold.loc[gold["query_id"] == qid, "product_id"].tolist()
    score_by_pid = {
        pid: float(scores[pid_to_idx[pid]])
        for pid in judged_pids if pid in pid_to_idx
    }

    ndcg, mrr = judged_metrics_for_scores(qid, score_by_pid, k=10)
    pos = known_positives.get(qid, set())

    bm25_query_metrics.append({
        "model": "BM25",
        "query_id": qid,
        "query": query,
        "NDCG@10_judged": ndcg,
        "MRR_judged": mrr,
        "Recall@10_full": recall_at_k(retrieved_pids, pos, 10),
        "Recall@50_full": recall_at_k(retrieved_pids, pos, 50),
    })

    if n % 25 == 0 or n == len(test_queries):
        print(f"BM25: {n}/{len(test_queries)} queries")

bm25_top50 = pd.DataFrame(bm25_top_rows)
bm25_qmetrics = pd.DataFrame(bm25_query_metrics)

print("\nBM25 Week 4 reference:")
display(pd.DataFrame([summarize_query_metrics(bm25_qmetrics)]))

In [ ]:
# ============================================================
# 9. SENTENCE-BERT PRODUCT EMBEDDINGS
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading SBERT model:", SBERT_MODEL)
print("Device:", device)

sbert = SentenceTransformer(SBERT_MODEL, device=device)
product_texts = products["retrieval_text"].fillna("").astype(str).tolist()

t0 = time.perf_counter()
product_embeddings = sbert.encode(
    product_texts,
    batch_size=SBERT_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")
product_encode_seconds = time.perf_counter() - t0

print("Product embedding shape:", product_embeddings.shape)
print("Product encoding time (sec):", round(product_encode_seconds, 2))
print("Approx embedding memory (MB):", round(product_embeddings.nbytes / 1024**2, 2))

np.save(OUTPUT_DIR / "week4_sbert_product_embeddings.npy", product_embeddings)

In [ ]:
# ============================================================
# 10. BUILD FAISS COSINE-SIMILARITY INDEX
# ============================================================

embedding_dim = product_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)

t0 = time.perf_counter()
index.add(product_embeddings)
faiss_build_ms = (time.perf_counter() - t0) * 1000.0

faiss.write_index(index, str(OUTPUT_DIR / "week4_sbert_faiss.index"))

print("FAISS index type:", type(index).__name__)
print("Vectors indexed:", index.ntotal)
print("Dimension:", embedding_dim)
print("Index build time (ms):", round(faiss_build_ms, 3))

In [ ]:
# ============================================================
# 11. SBERT + FAISS TOP-50 RETRIEVAL
# ============================================================

queries_list = test_queries["query"].tolist()

print("Encoding all test queries...")
query_embeddings = sbert.encode(
    queries_list,
    batch_size=SBERT_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

t0 = time.perf_counter()
faiss_scores, faiss_indices = index.search(query_embeddings, TOP_K)
batch_search_ms = (time.perf_counter() - t0) * 1000.0

sbert_top_rows = []
sbert_query_metrics = []

for qi, row in enumerate(test_queries.itertuples(index=False)):
    qid, query = row.query_id, row.query
    qemb = query_embeddings[qi]

    top_idx = faiss_indices[qi]
    top_scores = faiss_scores[qi]
    retrieved_pids = idx_to_pid[top_idx].tolist()

    for rank, (idx, score) in enumerate(zip(top_idx, top_scores), start=1):
        pid = str(idx_to_pid[idx])
        sbert_top_rows.append({
            "model": "SBERT+FAISS",
            "query_id": qid,
            "query": query,
            "rank": rank,
            "product_id": pid,
            "title": title_map.get(pid, ""),
            "relevance_label": relevance_lookup[qid].get(pid, np.nan),
            "score": float(score),
        })

    judged_pids = gold.loc[gold["query_id"] == qid, "product_id"].tolist()
    score_by_pid = {}
    for pid in judged_pids:
        idx = pid_to_idx.get(pid)
        if idx is not None:
            score_by_pid[pid] = float(np.dot(product_embeddings[idx], qemb))

    ndcg, mrr = judged_metrics_for_scores(qid, score_by_pid, k=10)
    pos = known_positives.get(qid, set())

    sbert_query_metrics.append({
        "model": "SBERT+FAISS",
        "query_id": qid,
        "query": query,
        "NDCG@10_judged": ndcg,
        "MRR_judged": mrr,
        "Recall@10_full": recall_at_k(retrieved_pids, pos, 10),
        "Recall@50_full": recall_at_k(retrieved_pids, pos, 50),
    })

sbert_top50 = pd.DataFrame(sbert_top_rows)
sbert_qmetrics = pd.DataFrame(sbert_query_metrics)

print("FAISS batch search total (ms):", round(batch_search_ms, 3))
print("FAISS batch search ms/query:", round(batch_search_ms / max(len(test_queries), 1), 4))

print("\nSBERT Week 4 metrics:")
display(pd.DataFrame([summarize_query_metrics(sbert_qmetrics)]))

In [ ]:
# ============================================================
# 12. SEQUENTIAL LATENCY BENCHMARK
# ============================================================

latency_queries = test_queries.head(min(LATENCY_SAMPLE_SIZE, len(test_queries))).copy()

if len(latency_queries):
    warm_q = latency_queries.iloc[0]["query"]
    warm_emb = sbert.encode(
        [warm_q],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")
    _ = index.search(warm_emb, TOP_K)

sbert_encode_latencies = []
faiss_search_latencies = []

for query in latency_queries["query"]:
    t0 = time.perf_counter()
    qemb = sbert.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype("float32")
    sbert_encode_latencies.append((time.perf_counter() - t0) * 1000.0)

    t0 = time.perf_counter()
    _ = index.search(qemb, TOP_K)
    faiss_search_latencies.append((time.perf_counter() - t0) * 1000.0)

def latency_summary(name, values):
    a = np.asarray(values, dtype=float)
    return {
        "component": name,
        "n_queries": len(a),
        "mean_ms": float(np.mean(a)),
        "median_ms": float(np.median(a)),
        "p95_ms": float(np.percentile(a, 95)),
        "max_ms": float(np.max(a)),
    }

latency_results = pd.DataFrame([
    latency_summary("BM25 full-catalog scoring + Top50", bm25_latency_ms[:len(latency_queries)]),
    latency_summary("SBERT query embedding", sbert_encode_latencies),
    latency_summary("FAISS Top50 search only", faiss_search_latencies),
    latency_summary(
        "SBERT embedding + FAISS search",
        np.asarray(sbert_encode_latencies) + np.asarray(faiss_search_latencies)
    ),
])

display(latency_results)

In [ ]:
# ============================================================
# 13. MODEL COMPARISON
# ============================================================

bm25_summary = summarize_query_metrics(bm25_qmetrics)
sbert_summary = summarize_query_metrics(sbert_qmetrics)

comparison = pd.DataFrame([
    {"model": "BM25", **bm25_summary},
    {"model": "SBERT+FAISS", **sbert_summary},
])

for metric in ["NDCG@10_judged", "MRR_judged", "Recall@10_full", "Recall@50_full"]:
    base = comparison.loc[comparison["model"] == "BM25", metric].iloc[0]
    comparison[f"{metric}_vs_BM25_%"] = np.where(
        base != 0,
        (comparison[metric] - base) / base * 100.0,
        np.nan
    )

display(comparison.round(4))

In [ ]:
# ============================================================
# 14. PER-QUERY DELTA / ERROR ANALYSIS
# ============================================================

per_query = bm25_qmetrics.merge(
    sbert_qmetrics,
    on=["query_id", "query"],
    suffixes=("_BM25", "_SBERT")
)

for metric in ["NDCG@10_judged", "MRR_judged", "Recall@10_full", "Recall@50_full"]:
    per_query[f"{metric}_delta"] = per_query[f"{metric}_SBERT"] - per_query[f"{metric}_BM25"]

best = per_query.sort_values("NDCG@10_judged_delta", ascending=False).head(10)
worst = per_query.sort_values("NDCG@10_judged_delta", ascending=True).head(10)

print("Top 10 SBERT gains:")
display(best[[
    "query_id", "query",
    "NDCG@10_judged_BM25", "NDCG@10_judged_SBERT",
    "NDCG@10_judged_delta",
    "Recall@50_full_BM25", "Recall@50_full_SBERT"
]])

print("\nTop 10 SBERT losses:")
display(worst[[
    "query_id", "query",
    "NDCG@10_judged_BM25", "NDCG@10_judged_SBERT",
    "NDCG@10_judged_delta",
    "Recall@50_full_BM25", "Recall@50_full_SBERT"
]])

In [ ]:
# ============================================================
# 15. QUICK QUALITATIVE DEMO
# ============================================================

def show_top_results(query_id, n=5):
    print("=" * 100)
    print("QUERY:", query_map.get(query_id, query_id))

    print("\nBM25")
    display(
        bm25_top50[bm25_top50["query_id"] == query_id]
        .head(n)[["rank", "product_id", "title", "relevance_label", "score"]]
    )

    print("\nSBERT + FAISS")
    display(
        sbert_top50[sbert_top50["query_id"] == query_id]
        .head(n)[["rank", "product_id", "title", "relevance_label", "score"]]
    )

for qid in test_queries["query_id"].head(3):
    show_top_results(qid, n=5)

In [ ]:
# ============================================================
# 16. PLOTS
# ============================================================

plot_metrics = ["NDCG@10_judged", "MRR_judged", "Recall@10_full", "Recall@50_full"]

ax = comparison.set_index("model")[plot_metrics].T.plot(kind="bar", figsize=(10, 5))
ax.set_title("Week 4 V1 — BM25 vs SBERT + FAISS")
ax.set_ylabel("Metric")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "week4_model_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

ax = latency_results.set_index("component")[["mean_ms", "p95_ms"]].plot(kind="bar", figsize=(10, 5))
ax.set_title("Week 4 V1 — Retrieval Latency")
ax.set_ylabel("Milliseconds")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "week4_latency_comparison.png", dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 17. EXPORT ALL WEEK 4 V1 ARTIFACTS
# ============================================================

bm25_top50.to_csv(OUTPUT_DIR / "week4_bm25_top50.csv", index=False)
sbert_top50.to_csv(OUTPUT_DIR / "week4_sbert_top50.csv", index=False)

bm25_qmetrics.to_csv(OUTPUT_DIR / "week4_bm25_query_metrics.csv", index=False)
sbert_qmetrics.to_csv(OUTPUT_DIR / "week4_sbert_query_metrics.csv", index=False)
per_query.to_csv(OUTPUT_DIR / "week4_per_query_comparison.csv", index=False)

comparison.to_csv(OUTPUT_DIR / "week4_text_retrieval_metrics.csv", index=False)
latency_results.to_csv(OUTPUT_DIR / "week4_latency_results.csv", index=False)
test_queries.to_csv(OUTPUT_DIR / "week4_test_queries_used.csv", index=False)

if week2_reference is not None:
    week2_reference.to_csv(OUTPUT_DIR / "week2_reference_metrics.csv", index=False)

readme = f'''
Amazon Search / LTR — Week 4 V1
================================

Pipeline
--------
Week 1 cleaned 10k catalog
    -> product_text
    -> Sentence-BERT ({SBERT_MODEL})
    -> L2-normalized embeddings
    -> FAISS IndexFlatIP
    -> cosine-similarity Top-{TOP_K}

Comparison
----------
BM25 full-catalog lexical retrieval
vs.
SBERT + FAISS semantic retrieval

Evaluation
----------
1) NDCG@10_judged:
   Ranks only Week 1 judged query-product candidates.

2) MRR_judged:
   Reciprocal rank of the first candidate with relevance_label > 0
   within the judged candidate set.

3) Recall@10_full / Recall@50_full:
   Full 10k catalog retrieval.
   Known relevant positives are Week 1 products with
   relevance_label >= {KNOWN_RELEVANT_THRESHOLD}.

Latency
-------
BM25: full-catalog scoring + Top-50
SBERT: query embedding
FAISS: Top-50 vector search only
SBERT+FAISS: embedding + search

Important
---------
This V1 establishes the text-only semantic retrieval baseline.
CLIP image embeddings and SBERT+CLIP multimodal fusion are Week 4 V2.
'''

with open(OUTPUT_DIR / "README_WEEK4_V1.txt", "w", encoding="utf-8") as f:
    f.write(readme)

zip_base = ROOT / "amazon_week4_v1_sbert_faiss_results"
zip_path = shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR)

print("Created ZIP:", zip_path)
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)

In [ ]:
# ============================================================
# 18. FINAL SCORECARD + AUTOMATIC DOWNLOAD
# ============================================================

print("\n" + "=" * 80)
print("WEEK 4 V1 COMPLETE")
print("=" * 80)

display(comparison.round(4))
display(latency_results.round(3))

sbert_row = comparison[comparison["model"] == "SBERT+FAISS"].iloc[0]
faiss_latency = latency_results[
    latency_results["component"] == "FAISS Top50 search only"
]["p95_ms"].iloc[0]

print("\nDevelopment checks")
print("------------------")
print(
    "FAISS P95 search < 50 ms:",
    "PASS" if faiss_latency < 50 else "FAIL",
    f"({faiss_latency:.3f} ms)"
)
print("SBERT Recall@50 vs BM25:", f'{sbert_row["Recall@50_full_vs_BM25_%"]:+.2f}%')
print("SBERT NDCG@10 vs BM25:", f'{sbert_row["NDCG@10_judged_vs_BM25_%"]:+.2f}%')
print("\nNext Week 4 step: add CLIP image embeddings + multimodal fusion.")

if is_colab:
    print("\nDownloading result ZIP...")
    files.download(str(ROOT / "amazon_week4_v1_sbert_faiss_results.zip"))
else:
    print("\nResult ZIP:", ROOT / "amazon_week4_v1_sbert_faiss_results.zip")